In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

df = pd.read_csv("../data/processed/fomc_spy.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)  # CV는 시간순 정렬 전제

tscv = TimeSeriesSplit(n_splits=5)  # Day 9-10과 동일한 expanding window

def eval_model(X, y, model_fn):
    # walk-forward: fold마다 과거로 train, 바로 다음 구간으로 test
    oos_true, oos_pred = [], []
    for tr, te in tscv.split(X):
        m = model_fn().fit(X[tr], y[tr])
        oos_pred.extend(m.predict(X[te]))
        oos_true.extend(y[te])
    return r2_score(oos_true, oos_pred)  # pooled out-of-sample R²

X = df[["sentiment"]].to_numpy()  # sentiment 단독, Day 9-10과 동일 피처
for tgt in ["ret_next", "ret"]:
    y = df[tgt].to_numpy()
    rf  = eval_model(X, y, lambda: RandomForestRegressor(
                     n_estimators=200, max_depth=3, random_state=42))
    dm  = eval_model(X, y, lambda: DummyRegressor(strategy="mean"))
    print(f"{tgt:9s}  RF R2={rf:+.3f}   Dummy R2={dm:+.3f}")

ret_next   RF R2=-0.359   Dummy R2=-0.042
ret        RF R2=-0.327   Dummy R2=-0.015


In [2]:
# --- feature engineering ---
# word count of clean_text = document length (absolute signal)
df["doc_len"] = df["clean_text"].str.split().str.len()
# sentiment change vs previous meeting (first difference); past values only -> look-ahead safe
df["sent_delta"] = df["sentiment"].diff()
# regime label: reuse same definition as EDA Part C (sentiment sign threshold)
df["regime"] = np.where(df["sentiment"] > 0.05, "hawkish",
                np.where(df["sentiment"] < -0.05, "dovish", "neutral"))
reg = pd.get_dummies(df["regime"], prefix="reg", drop_first=True)  # 2 dummies, avoid trap
df = pd.concat([df, reg.astype(int)], axis=1)

df_fe = df.dropna(subset=["sent_delta"]).reset_index(drop=True)  # drop first row NaN only

feat_cols = ["sentiment", "doc_len", "sent_delta"] + list(reg.columns)
print("features:", feat_cols, "| rows:", len(df_fe))

X_fe = df_fe[feat_cols].to_numpy()
for tgt in ["ret_next", "ret"]:
    y = df_fe[tgt].to_numpy()
    rf = eval_model(X_fe, y, lambda: RandomForestRegressor(
                    n_estimators=200, max_depth=3, random_state=42))
    dm = eval_model(X_fe, y, lambda: DummyRegressor(strategy="mean"))
    print(f"{tgt:9s}  RF(multi) R2={rf:+.3f}   Dummy R2={dm:+.3f}")

features: ['sentiment', 'doc_len', 'sent_delta', 'reg_hawkish', 'reg_neutral'] | rows: 124
ret_next   RF(multi) R2=-0.287   Dummy R2=-0.046
ret        RF(multi) R2=-0.307   Dummy R2=-0.014


In [ ]:
# fit on full FE data just to inspect importances
rf_full = RandomForestRegressor(n_estimators=200, max_depth=3,
                                random_state=42).fit(X_fe, df_fe["ret_next"].to_numpy())
imp = pd.Series(rf_full.feature_importances_, index=feat_cols).sort_values(ascending=False)
print(imp)

sent_delta     0.532547
doc_len        0.288243
sentiment      0.171549
reg_hawkish    0.005922
reg_neutral    0.001739
dtype: float64
